# Streaming Data Analysis

**Dataset:** Netflix Customer Churn Dataset (Kaggle)  
**Domain:** Video streaming / subscription analytics  
**Tools:** Python, Pandas, NumPy, Matplotlib, Seaborn, Scikit-learn

This notebook follows the project tasks in `tasks.md` using a streaming-domain dataset. The original rubric uses retail-style fields like *Sales* and *Profit*, so this notebook adapts them as follows:

- `Sales` equivalent -> `watch_hours`
- engagement predictors -> `avg_watch_time_per_day`, `last_login_days`, `monthly_fee`, `number_of_profiles`
- classification target -> `churned`


In [ ]:
!pip install -q kagglehub pandas numpy matplotlib seaborn scikit-learn

import kagglehub
from pathlib import Path

path = Path(kagglehub.dataset_download("abdulwadood11220/netflix-customer-churn-dataset"))
print(path)


In [ ]:
import io
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, PolynomialFeatures, StandardScaler

sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (8, 5)

csv_path = path / 'netflix_customer_churn.csv'
df = pd.read_csv(csv_path)

REGRESSION_FEATURES = ['age', 'last_login_days', 'monthly_fee', 'number_of_profiles', 'avg_watch_time_per_day']
CLASSIFICATION_NUMERIC_FEATURES = ['age', 'watch_hours', 'last_login_days', 'monthly_fee', 'number_of_profiles', 'avg_watch_time_per_day']
CLASSIFICATION_CATEGORICAL_FEATURES = ['gender', 'subscription_type', 'region', 'device', 'payment_method', 'favorite_genre']
NUMERIC_COLUMNS = ['age', 'watch_hours', 'last_login_days', 'monthly_fee', 'churned', 'number_of_profiles', 'avg_watch_time_per_day']

df.head()


## Task 1: Data Understanding

The dataset is loaded using Pandas. This section displays the first 5 rows, last 5 rows, shape, column names, data types, and summary statistics.


In [ ]:
print('First 5 rows:')
display(df.head())

print('Last 5 rows:')
display(df.tail())

print('Dataset shape:', df.shape)
print('Column names:', list(df.columns))


In [ ]:
buffer = io.StringIO()
df.info(buf=buffer)
print(buffer.getvalue())

display(df[NUMERIC_COLUMNS].describe().round(3))


In [ ]:
quantitative_discrete = ['age', 'last_login_days', 'number_of_profiles', 'churned']
quantitative_continuous = ['watch_hours', 'monthly_fee', 'avg_watch_time_per_day']
qualitative_nominal = ['customer_id', 'gender', 'region', 'device', 'payment_method', 'favorite_genre']
qualitative_ordinal = ['subscription_type']

print('Quantitative (Discrete):', quantitative_discrete)
print('Quantitative (Continuous):', quantitative_continuous)
print('Qualitative (Nominal):', qualitative_nominal)
print('Qualitative (Ordinal):', qualitative_ordinal)


## Reusable EDA Functions

These helper functions are used throughout the notebook to automate repeated EDA tasks.


In [ ]:
def plot_histogram_and_boxplot(dataframe, column, bins=25):
    fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
    sns.histplot(dataframe[column], kde=True, bins=bins, ax=axes[0], color='#2563eb')
    axes[0].set_title(f'Histogram of {column}')
    sns.boxplot(x=dataframe[column], ax=axes[1], color='#f59e0b')
    axes[1].set_title(f'Boxplot of {column}')
    plt.tight_layout()
    plt.show()

def plot_countplot(dataframe, column):
    plt.figure(figsize=(10, 5))
    order = dataframe[column].value_counts().index
    ax = sns.countplot(data=dataframe, x=column, order=order, hue=column, palette='Blues_r', legend=False)
    plt.title(f'Count Plot of {column}')
    plt.xticks(rotation=25, ha='right')
    if ax.get_legend() is not None:
        ax.get_legend().remove()
    plt.tight_layout()
    plt.show()

def plot_scatter(dataframe, x_col, y_col):
    plt.figure(figsize=(8, 5))
    sns.scatterplot(data=dataframe, x=x_col, y=y_col, hue='churned', alpha=0.65, palette='Set1')
    plt.title(f'{y_col} vs {x_col}')
    plt.tight_layout()
    plt.show()

def outlier_summary(dataframe, columns):
    rows = []
    for column in columns:
        q1 = dataframe[column].quantile(0.25)
        q3 = dataframe[column].quantile(0.75)
        iqr = q3 - q1
        lower_bound = q1 - 1.5 * iqr
        upper_bound = q3 + 1.5 * iqr
        mask = (dataframe[column] < lower_bound) | (dataframe[column] > upper_bound)
        rows.append({
            'column': column,
            'q1': q1,
            'q3': q3,
            'iqr': iqr,
            'lower_bound': lower_bound,
            'upper_bound': upper_bound,
            'outlier_count': int(mask.sum()),
            'outlier_percentage': round(mask.mean() * 100, 2)
        })
    return pd.DataFrame(rows)

def distribution_statistics(dataframe, columns):
    return pd.DataFrame({
        'mean': dataframe[columns].mean(),
        'median': dataframe[columns].median(),
        'std_dev': dataframe[columns].std(),
        'skewness': dataframe[columns].skew(),
        'kurtosis': dataframe[columns].kurt()
    })


## Task 2: Exploratory Data Analysis (EDA)

This section covers univariate, bivariate, and multivariate analysis using streaming-domain variables.


In [ ]:
for column in ['watch_hours', 'monthly_fee', 'avg_watch_time_per_day', 'last_login_days']:
    plot_histogram_and_boxplot(df, column)

for column in ['subscription_type', 'device', 'favorite_genre', 'churned']:
    plot_countplot(df, column)


In [ ]:
plot_scatter(df, 'avg_watch_time_per_day', 'watch_hours')
plot_scatter(df, 'last_login_days', 'watch_hours')
plot_scatter(df, 'monthly_fee', 'watch_hours')

corr_matrix = df[NUMERIC_COLUMNS].corr(numeric_only=True)
display(corr_matrix.round(3))

plt.figure(figsize=(8, 6))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt='.2f', square=True)
plt.title('Correlation Heatmap')
plt.tight_layout()
plt.show()

sampled = df[['watch_hours', 'last_login_days', 'monthly_fee', 'avg_watch_time_per_day', 'churned']].sample(n=min(600, len(df)), random_state=42)
sns.pairplot(sampled, vars=['watch_hours', 'last_login_days', 'monthly_fee', 'avg_watch_time_per_day'], hue='churned', corner=True, diag_kind='hist')
plt.show()

plt.figure(figsize=(10, 5))
ax = sns.boxplot(data=df, x='subscription_type', y='watch_hours', hue='subscription_type', palette='Set3', dodge=False)
if ax.get_legend() is not None:
    ax.get_legend().remove()
plt.title('Watch Hours across Subscription Type')
plt.tight_layout()
plt.show()

plt.figure(figsize=(11, 5))
ax = sns.boxplot(data=df, x='favorite_genre', y='watch_hours', hue='favorite_genre', palette='Set2', dodge=False)
if ax.get_legend() is not None:
    ax.get_legend().remove()
plt.title('Watch Hours across Favorite Genre')
plt.xticks(rotation=25, ha='right')
plt.tight_layout()
plt.show()

plt.figure(figsize=(11, 5))
ax = sns.boxplot(data=df, x='region', y='watch_hours', hue='region', palette='Set1', dodge=False)
if ax.get_legend() is not None:
    ax.get_legend().remove()
plt.title('Watch Hours across Region')
plt.xticks(rotation=25, ha='right')
plt.tight_layout()
plt.show()


## Task 3: Handling Missing Data and Outliers

The raw dataset is kept unchanged. Missing values are checked honestly, and outliers are detected using the IQR rule and boxplots.


In [ ]:
missing_values = df.isnull().sum()
display(missing_values.to_frame(name='missing_count'))

if missing_values.sum() == 0:
    print('No missing values were found in the raw dataset, so no imputation was applied.')
    print('If missing values existed, numeric columns would be handled with mean or median and categorical columns with mode.')

outliers_df = outlier_summary(df, ['age', 'watch_hours', 'last_login_days', 'monthly_fee', 'number_of_profiles', 'avg_watch_time_per_day'])
display(outliers_df.round(3))

print('Impact of outliers: extreme engagement values can distort averages, increase variance, and influence regression coefficients.')


## Task 4: Spread of Data

This section measures central tendency and shape of the data using mean, median, standard deviation, skewness, and kurtosis.

## Task 5: Automating EDA using Python

The notebook uses `describe()`, `info()`, `isnull()`, and `corr()` directly, while the helper functions automate repeated visual analysis.


In [ ]:
distribution_df = distribution_statistics(df, ['age', 'watch_hours', 'last_login_days', 'monthly_fee', 'number_of_profiles', 'avg_watch_time_per_day'])
display(distribution_df.round(4))

print('Interpretation: skewness near 0 suggests a more symmetric distribution, while larger positive or negative values indicate skewness.')
print('Kurtosis helps identify heavier tails or sharper peaks relative to a normal distribution.')

display(df[NUMERIC_COLUMNS].corr().round(3))


## Task 6: Regression Analysis

- Dependent variable: `watch_hours`
- Simple linear regression feature: `avg_watch_time_per_day`
- Multiple linear regression features: `age`, `last_login_days`, `monthly_fee`, `number_of_profiles`, `avg_watch_time_per_day`

## Task 7: Supervised Learning - Regression Model

The dataset is split into training, validation, and testing sets. We then build simple linear regression, multiple linear regression, and logistic regression models.


In [ ]:
covariance_value = df['watch_hours'].cov(df['avg_watch_time_per_day'])
correlation_value = df['watch_hours'].corr(df['avg_watch_time_per_day'])

print(f'Covariance between watch_hours and avg_watch_time_per_day: {covariance_value:.4f}')
print(f'Correlation between watch_hours and avg_watch_time_per_day: {correlation_value:.4f}')

train_df, temp_df = train_test_split(df, test_size=0.40, random_state=42, stratify=df['churned'])
validation_df, test_df = train_test_split(temp_df, test_size=0.50, random_state=42, stratify=temp_df['churned'])

print('Training set size:', train_df.shape)
print('Validation set size:', validation_df.shape)
print('Testing set size:', test_df.shape)

y_train_reg = train_df['watch_hours']
y_val_reg = validation_df['watch_hours']
y_test_reg = test_df['watch_hours']

simple_model = LinearRegression()
simple_model.fit(train_df[['avg_watch_time_per_day']], y_train_reg)

def regression_metrics(y_true, predictions):
    return {
        'MSE': mean_squared_error(y_true, predictions),
        'MAE': mean_absolute_error(y_true, predictions),
        'R2': r2_score(y_true, predictions)
    }

simple_results = []
for split_name, features, target in [
    ('train', train_df[['avg_watch_time_per_day']], y_train_reg),
    ('validation', validation_df[['avg_watch_time_per_day']], y_val_reg),
    ('test', test_df[['avg_watch_time_per_day']], y_test_reg),
]:
    preds = simple_model.predict(features)
    simple_results.append({'model': 'simple_linear_regression', 'split': split_name, **regression_metrics(target, preds)})

multi_model = LinearRegression()
multi_model.fit(train_df[REGRESSION_FEATURES], y_train_reg)

multi_results = []
for split_name, features, target in [
    ('train', train_df[REGRESSION_FEATURES], y_train_reg),
    ('validation', validation_df[REGRESSION_FEATURES], y_val_reg),
    ('test', test_df[REGRESSION_FEATURES], y_test_reg),
]:
    preds = multi_model.predict(features)
    multi_results.append({'model': 'multiple_linear_regression', 'split': split_name, **regression_metrics(target, preds)})

regression_results_df = pd.DataFrame(simple_results + multi_results)
display(regression_results_df.round(6))

plot_df = test_df[['avg_watch_time_per_day']].copy()
plot_df['actual_watch_hours'] = y_test_reg.to_numpy()
plot_df['predicted_watch_hours'] = simple_model.predict(test_df[['avg_watch_time_per_day']])
plot_df = plot_df.sort_values(by='avg_watch_time_per_day')

plt.figure(figsize=(7, 5))
sns.scatterplot(data=plot_df, x='avg_watch_time_per_day', y='actual_watch_hours', label='Actual', alpha=0.6)
plt.plot(plot_df['avg_watch_time_per_day'], plot_df['predicted_watch_hours'], color='red', label='Predicted')
plt.title('Simple Linear Regression Fit on Test Data')
plt.xlabel('avg_watch_time_per_day')
plt.ylabel('watch_hours')
plt.legend()
plt.tight_layout()
plt.show()


## Task 8: Overfitting and Underfitting Analysis

A very simple model may underfit the data, while a very complex model may overfit and perform poorly on unseen data. We compare train and test MSE across polynomial degrees.


In [ ]:
complexity_rows = []
for degree in [1, 2, 3, 5]:
    poly_model = Pipeline([
        ('poly', PolynomialFeatures(degree=degree, include_bias=False)),
        ('linear', LinearRegression())
    ])
    poly_model.fit(train_df[['avg_watch_time_per_day']], y_train_reg)
    train_pred = poly_model.predict(train_df[['avg_watch_time_per_day']])
    test_pred = poly_model.predict(test_df[['avg_watch_time_per_day']])
    complexity_rows.append({
        'degree': degree,
        'train_mse': mean_squared_error(y_train_reg, train_pred),
        'test_mse': mean_squared_error(y_test_reg, test_pred)
    })

complexity_df = pd.DataFrame(complexity_rows)
display(complexity_df.round(6))

plt.figure(figsize=(8, 5))
plt.plot(complexity_df['degree'], complexity_df['train_mse'], marker='o', label='Train MSE')
plt.plot(complexity_df['degree'], complexity_df['test_mse'], marker='o', label='Test MSE')
plt.title('Model Complexity vs Error')
plt.xlabel('Polynomial Degree')
plt.ylabel('Mean Squared Error')
plt.legend()
plt.tight_layout()
plt.show()

print('Underfitting occurs when the model is too simple and performs poorly on both train and test data.')
print('Overfitting occurs when training error becomes much lower than testing error.')


## Task 9: Classification Task

The problem is converted into a classification problem using `churned` as the target variable. Logistic regression is used to predict whether a customer will churn.


In [ ]:
y_train_cls = train_df['churned']
y_val_cls = validation_df['churned']
y_test_cls = test_df['churned']

preprocessor = ColumnTransformer([
    ('num', StandardScaler(), CLASSIFICATION_NUMERIC_FEATURES),
    ('cat', OneHotEncoder(handle_unknown='ignore'), CLASSIFICATION_CATEGORICAL_FEATURES)
])

logistic_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(max_iter=1000))
])

feature_columns = CLASSIFICATION_NUMERIC_FEATURES + CLASSIFICATION_CATEGORICAL_FEATURES
logistic_pipeline.fit(train_df[feature_columns], y_train_cls)

classification_rows = []
for split_name, features_df, target in [
    ('train', train_df, y_train_cls),
    ('validation', validation_df, y_val_cls),
    ('test', test_df, y_test_cls),
]:
    predictions = logistic_pipeline.predict(features_df[feature_columns])
    classification_rows.append({
        'model': 'logistic_regression',
        'split': split_name,
        'accuracy': accuracy_score(target, predictions)
    })

classification_df = pd.DataFrame(classification_rows)
display(classification_df.round(6))

validation_predictions = logistic_pipeline.predict(validation_df[feature_columns])
test_predictions = logistic_pipeline.predict(test_df[feature_columns])
validation_cm = confusion_matrix(y_val_cls, validation_predictions)
test_cm = confusion_matrix(y_test_cls, test_predictions)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
sns.heatmap(validation_cm, annot=True, fmt='d', cmap='Blues', cbar=False, ax=axes[0])
axes[0].set_title('Validation Confusion Matrix')
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('Actual')

sns.heatmap(test_cm, annot=True, fmt='d', cmap='Blues', cbar=False, ax=axes[1])
axes[1].set_title('Test Confusion Matrix')
axes[1].set_xlabel('Predicted')
axes[1].set_ylabel('Actual')

plt.tight_layout()
plt.show()

display(pd.DataFrame(test_cm, index=['Actual_0', 'Actual_1'], columns=['Pred_0', 'Pred_1']))


## Task 10 and Task 11: Model Evaluation

Regression models are evaluated using MSE, MAE, and R2 score. Classification performance is evaluated using accuracy and confusion matrix.

## Task 12: Data Visualization Summary

All required univariate, bivariate, multivariate, and distribution plots have been generated inline in this notebook.


In [ ]:
best_regression_test = regression_results_df[regression_results_df['split'] == 'test'].sort_values(by='R2', ascending=False)
best_classification_test = classification_df[classification_df['split'] == 'test']

print('Regression results on test data:')
display(best_regression_test.round(6))

print('Classification results on test data:')
display(best_classification_test.round(6))

print('Comparison: regression estimates user engagement hours, while classification predicts churn risk.')

watch_hours_by_churn = df.groupby('churned')['watch_hours'].mean().rename('mean_watch_hours')
avg_login_gap_by_churn = df.groupby('churned')['last_login_days'].mean().rename('mean_last_login_days')
display(pd.concat([watch_hours_by_churn, avg_login_gap_by_churn], axis=1).round(3))


## Conclusion

This notebook completes the full streaming data analysis workflow using the Netflix Customer Churn dataset. It covers:

- data understanding and variable classification
- univariate, bivariate, and multivariate EDA
- missing value checks and outlier analysis
- spread and distribution statistics
- simple and multiple linear regression
- logistic regression for churn prediction
- model evaluation using MSE, MAE, R2, accuracy, and confusion matrix

You can now use this notebook directly for screenshots, observations, and your final report.
